<a href="https://colab.research.google.com/github/22495521/99-multiplication-table/blob/main/faster_rcnn_vehicles_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Faster R-CNN 車輛偵測（Google Colab 版）

> 這是給 **Google Colab** 用的版本。開始前先到選單 **執行階段 → 變更執行階段類型 → 硬體加速器選 GPU（T4）**，
> 不然用 CPU 訓練會非常慢。

參考：[TorchVision Object Detection Finetuning Tutorial](https://docs.pytorch.org/tutorials/intermediate/torchvision_tutorial.html)

改自 PennFudan 行人偵測版本，換成自己的車輛資料集 `vehicles_images/`（5 類：Bus、Car、Motorcycle、Pickup、Truck）。

**和 PennFudan 版本的差異**

| | PennFudan | vehicles_images |
|---|---|---|
| 標註格式 | 實例遮罩 PNG → `masks_to_boxes` 轉框 | CSV 直接給 `xmin, ymin, xmax, ymax` |
| 類別數 | 1 類（行人） | 5 類（車種），加背景共 6 |
| 訓練 / 測試切分 | 自己隨機切 | 已經分好 `train/`、`test/` |
| 資料來源 | 自動下載 | 從 Google Drive 或手動上傳 `vehicles_images.zip` |
| 沒有標註的圖 | 無 | `train/` 有 2 張沒有任何框，當作純背景圖（負樣本） |

**執行流程**

0. Colab 環境：檢查 GPU、安裝套件、把資料集放到 `/content/vehicles_images`。
1. 準備資料：讀 `train_labels.csv` / `test_labels.csv`，依檔名把同一張圖的多個框分組。
2. 定義 Dataset：組成 Faster R-CNN 要的 target 格式。
3. 建立模型：載入在 COCO 預訓練好的 Faster R-CNN，把分類頭換成 6 類（背景 + 5 種車）。
4. 訓練：微調（fine-tune）數個 epoch（用官方 `engine.py` 的 `train_one_epoch`）。
5. 評估：在測試集上算 COCO mAP（用官方 `engine.py` 的 `evaluate`，需要 `pycocotools`）。
6. 推論：對測試圖畫出預測框，存到 `outputs/`（有掛載 Drive 時存到 Drive，斷線也不會不見）。

**套件**：Colab 已內建 `torch`、`torchvision`、`matplotlib`，只需確認 `pycocotools`。

## 0-1. 檢查 GPU、安裝套件

In [ ]:
!nvidia-smi -L
!pip install -q pycocotools

import torch
print("torch", torch.__version__, "| CUDA 可用：", torch.cuda.is_available())

## 0-2. 準備資料集

先在自己電腦把 `vehicles_images` 資料夾壓成 **`vehicles_images.zip`**
（zip 打開後第一層要是 `vehicles_images/`，裡面有 `train/`、`test/`、`train_labels.csv`、`test_labels.csv`）。

兩種放法擇一，改 `DATA_SOURCE` 就好：

- `"drive"`（推薦）：把 zip 上傳到 Google Drive 的 `MyDrive/rcnn/vehicles_images.zip`，
  掛載後自動解壓。好處是只要傳一次，訓練好的權重也會存回 Drive。
- `"upload"`：每次執行時從電腦選檔上傳（Colab 斷線後 `/content` 會被清空，要重傳）。

> 不建議直接在 Drive 裡讀散開的圖片，Drive 讀小檔很慢；複製 zip 到 `/content` 再解壓快很多。

In [ ]:
import shutil
import zipfile
from pathlib import Path

DATA_SOURCE = "drive"  # "drive" 或 "upload"
DRIVE_DIR = Path("/content/drive/MyDrive/rcnn")  # Drive 上放 zip、存權重的資料夾
ZIP_NAME = "vehicles_images.zip"

CONTENT = Path("/content")
DATA_DIR = CONTENT / "vehicles_images"

if DATA_SOURCE == "drive":
    from google.colab import drive

    drive.mount("/content/drive")
    zip_path = DRIVE_DIR / ZIP_NAME
    assert zip_path.exists(), f"找不到 {zip_path}，請先把 zip 上傳到 Drive 的這個位置"
    OUTPUT_DIR = DRIVE_DIR / "outputs"  # 權重存回 Drive
else:
    from google.colab import files

    uploaded = files.upload()  # 跳出選檔視窗，選 vehicles_images.zip
    zip_path = CONTENT / next(iter(uploaded))
    OUTPUT_DIR = CONTENT / "outputs"  # 斷線會消失，記得自己下載權重

if not DATA_DIR.exists():
    local_zip = CONTENT / ZIP_NAME
    if zip_path != local_zip:
        shutil.copy(zip_path, local_zip)  # 先複製到本機磁碟，解壓比直接讀 Drive 快
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall(CONTENT)

# 防呆：如果 zip 裡沒有包一層 vehicles_images/，檔案會直接散在 /content
if not DATA_DIR.exists() and (CONTENT / "train_labels.csv").exists():
    DATA_DIR.mkdir()
    for name in ["train", "test", "train_labels.csv", "test_labels.csv", "category.txt"]:
        if (CONTENT / name).exists():
            shutil.move(str(CONTENT / name), DATA_DIR / name)

for name in ["train", "test", "train_labels.csv", "test_labels.csv"]:
    assert (DATA_DIR / name).exists(), f"{DATA_DIR / name} 不存在，請檢查 zip 結構"
print("資料集位置：", DATA_DIR)
print("輸出位置：", OUTPUT_DIR)

## 0-3. 下載官方輔助檔案

`engine.py`、`utils.py`、`coco_utils.py`、`coco_eval.py`、`transforms.py` 來自
[torchvision references/detection](https://github.com/pytorch/vision/tree/main/references/detection)，
下載到 `/content`（Colab 的工作目錄），之後才能 `from engine import ...`。

In [ ]:
import os
import urllib.request

os.chdir("/content")

REF_URL = "https://raw.githubusercontent.com/pytorch/vision/main/references/detection/"
for name in ["engine.py", "utils.py", "coco_utils.py", "coco_eval.py", "transforms.py"]:
    if not Path(name).exists():
        print(f"下載 {name}")
        urllib.request.urlretrieve(REF_URL + name, name)
print("輔助檔案都準備好了")

## 1. 載入套件

In [ ]:
import csv
import random
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torchvision import tv_tensors
from torchvision.io import ImageReadMode, read_image
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import v2 as T
from torchvision.transforms.v2 import functional as F
from torchvision.utils import draw_bounding_boxes

# 官方 torchvision references/detection 的訓練 / 評估函式
# train_one_epoch：訓練一個 epoch，第 0 個 epoch 有做 warmup，loss 變 nan 會自動停止
# evaluate       ：用 pycocotools 算 COCO mAP
from engine import evaluate, train_one_epoch

## 2. 參數設定

`category.txt` 裡是 `{'Bus':0, 'Car':1, 'Motorcycle':2, 'Pickup':3, 'Truck':4}`，
但 **Faster R-CNN 規定 label 0 一定是背景**，所以這裡全部往後移一號：Bus=1 … Truck=5。

In [ ]:
# DATA_DIR、OUTPUT_DIR 已在 0-2 節設定好

# 0 保留給背景，車種從 1 開始
CLASSES = ["__background__", "Bus", "Car", "Motorcycle", "Pickup", "Truck"]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)  # 6 = 背景 + 5 種車

# 畫框用的顏色，每個類別一個
CLASS_COLORS = {"Bus": "orange", "Car": "lime", "Motorcycle": "cyan", "Pickup": "magenta", "Truck": "yellow"}

NUM_EPOCHS = 10  # 5 類比 1 類難，訓練圖也只有 136 張，多跑幾個 epoch
BATCH_SIZE = 2
SCORE_THRESHOLD = 0.5  # 視覺化時只畫信心分數 > 0.5 的框
SEED = 42

torch.manual_seed(SEED)
random.seed(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 有 GPU 用 GPU，沒有就用 CPU（Colab 記得開 GPU 執行階段）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用裝置：{device}")

## 3. Dataset 定義

資料夾結構：

```
vehicles_images/
├── train/              訓練圖（640x640 jpg）
├── test/               測試圖
├── train_labels.csv    一列一個框：filename,width,height,class,xmin,ymin,xmax,ymax
└── test_labels.csv
```

一張圖可能有好幾個框，所以 CSV 會有好幾列同一個 `filename`，要先依檔名分組。
圖片清單以**資料夾裡實際的檔案**為準，這樣 CSV 裡沒出現的圖（沒有任何車）也會被讀進來當負樣本。

`__getitem__` 回傳 `(image, target)`，target 是 Faster R-CNN 規定的 dict：

| 欄位 | 說明 |
|---|---|
| `boxes` | `[N, 4]` 浮點數，格式為 `(x1, y1, x2, y2)`；沒有框時是 `[0, 4]` |
| `labels` | `[N]` 整數，每個框的類別（1~5） |
| `image_id` | 圖片編號 |
| `area` | 每個框的面積 |
| `iscrowd` | 是否為群體標註（這裡全為 0） |

In [ ]:
class VehicleDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, csv_path, transforms=None):
        self.img_dir = Path(img_dir)
        self.transforms = transforms
        # 排序確保每次順序一致
        self.imgs = sorted(p for p in self.img_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})

        # 依檔名分組：{filename: [(x1, y1, x2, y2, label), ...]}
        self.annotations = defaultdict(list)
        with open(csv_path, newline="") as f:
            for row in csv.DictReader(f):
                box = [float(row[k]) for k in ("xmin", "ymin", "xmax", "ymax")]
                self.annotations[row["filename"]].append((*box, CLASS_TO_IDX[row["class"]]))

    def __getitem__(self, idx):
        img_path = self.imgs[idx]
        # 讀圖：shape 為 [3, H, W]，dtype uint8
        img = read_image(str(img_path), mode=ImageReadMode.RGB)

        anns = self.annotations.get(img_path.name, [])
        if anns:
            boxes = torch.tensor([a[:4] for a in anns], dtype=torch.float32)
            labels = torch.tensor([a[4] for a in anns], dtype=torch.int64)
        else:
            # 沒有任何車的圖：給空的框，模型會把整張圖當背景學
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)

        target = {
            # 用 tv_tensors.BoundingBoxes 包裝，做翻轉等資料增強時框會跟著圖一起轉換
            "boxes": tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=F.get_size(img)),
            "labels": labels,
            "image_id": idx,
            "area": (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]),
            "iscrowd": torch.zeros((len(labels),), dtype=torch.int64),
        }
        img = tv_tensors.Image(img)

        if self.transforms is not None:
            img, target = self.transforms(img, target)
        return img, target

    def __len__(self):
        return len(self.imgs)

### 資料前處理與 collate_fn

In [ ]:
def get_transform(train):
    """資料前處理 / 資料增強。"""
    transforms = []
    if train:
        # 訓練時 50% 機率水平翻轉，增加資料多樣性（框會自動跟著翻）
        transforms.append(T.RandomHorizontalFlip(0.5))
    # uint8 [0, 255] → float32 [0, 1]；模型內部會再做 normalize 與 resize
    transforms.append(T.ToDtype(torch.float, scale=True))
    # 把 tv_tensors 轉回一般 tensor
    transforms.append(T.ToPureTensor())
    return T.Compose(transforms)


def collate_fn(batch):
    """
    每張圖的框數量不同，不能直接 stack 成一個 tensor，
    所以把一個 batch 打包成 (圖片的 tuple, target 的 tuple)。
    """
    return tuple(zip(*batch))

### 建立 DataLoader

資料集已經分好 `train/` 和 `test/`，不用再自己隨機切。

In [ ]:
dataset = VehicleDataset(DATA_DIR / "train", DATA_DIR / "train_labels.csv", get_transform(train=True))
dataset_test = VehicleDataset(DATA_DIR / "test", DATA_DIR / "test_labels.csv", get_transform(train=False))
print(f"訓練集 {len(dataset)} 張，測試集 {len(dataset_test)} 張")

# 統計各類別框的數量，看一下類別是否平衡
for name, ds in [("train", dataset), ("test", dataset_test)]:
    counts = defaultdict(int)
    for anns in ds.annotations.values():
        for a in anns:
            counts[CLASSES[a[4]]] += 1
    print(name, dict(sorted(counts.items())))

# notebook（尤其 Windows / macOS）上多進程讀資料容易出問題，num_workers 設 0 最保險
data_loader = torch.utils.data.DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_fn
)
data_loader_test = torch.utils.data.DataLoader(
    dataset_test, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_fn
)

### 檢查一筆資料

In [ ]:
img, target = dataset[0]
print("image:", img.shape, img.dtype)
for k, v in target.items():
    print(f"{k}:", v.shape if isinstance(v, torch.Tensor) else v)

names = [CLASSES[i] for i in target["labels"].tolist()]
drawn = draw_bounding_boxes(
    (img * 255).to(torch.uint8), target["boxes"], names, colors=[CLASS_COLORS[n] for n in names], width=2
)
plt.figure(figsize=(7, 7))
plt.imshow(drawn.permute(1, 2, 0))
plt.axis("off")
plt.show()

## 4. 模型

載入在 COCO 上預訓練的 Faster R-CNN（ResNet-50 + FPN 骨幹網路），
再把最後的「框分類 / 框回歸」頭換成我們自己的類別數（6）。
這就是遷移學習：骨幹網路、RPN 都沿用預訓練權重，只需要微調。

In [ ]:
def get_model(num_classes):
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
    # 取得原本分類頭的輸入維度（1024）
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    # 換成新的預測頭：輸出 num_classes 個類別分數 + 每類 4 個框座標偏移
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


model = get_model(NUM_CLASSES).to(device)

## 5. 優化器與學習率排程（與官方教學相同）

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
# 每 3 個 epoch 學習率乘 0.1
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

## 6. 訓練與評估

每個 epoch 結束會印出 COCO mAP 表格，第二行 `IoU=0.50` 那個就是 AP@0.5。
注意 COCO mAP 是 5 個類別的平均，Bus、Truck 樣本很少，分數會被拉低。

In [ ]:
for epoch in range(NUM_EPOCHS):
    # print_freq=10：每 10 個 iteration 印一次 loss
    train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10)
    lr_scheduler.step()
    evaluate(model, data_loader_test, device=device)

## 7. 儲存權重

In [ ]:
weight_path = OUTPUT_DIR / "fasterrcnn_vehicles.pth"
torch.save(model.state_dict(), weight_path)
print(f"模型權重已存到 {weight_path}")

# 用上傳模式時 /content 斷線就會清空，順便下載一份到自己電腦
if DATA_SOURCE == "upload":
    from google.colab import files

    files.download(str(weight_path))

## 8. 推論與視覺化

對單張圖片做推論，把信心分數高於門檻的框畫上去（每個類別不同顏色），顯示在 notebook 並存檔。

In [ ]:
@torch.no_grad()
def predict_and_show(model, image_path, device, save_path=None):
    model.eval()
    image = read_image(str(image_path), mode=ImageReadMode.RGB)
    x = get_transform(train=False)(image).to(device)
    pred = model([x])[0]

    keep = pred["scores"] > SCORE_THRESHOLD
    boxes = pred["boxes"][keep].cpu()
    names = [CLASSES[i] for i in pred["labels"][keep].tolist()]
    scores = pred["scores"][keep].tolist()
    labels = [f"{n}: {s:.2f}" for n, s in zip(names, scores)]

    drawn = draw_bounding_boxes(image, boxes, labels, colors=[CLASS_COLORS[n] for n in names], width=2)
    counts = {n: names.count(n) for n in CLASSES[1:] if n in names}
    print(f"偵測到 {len(boxes)} 台車：{counts}")

    plt.figure(figsize=(8, 8))
    plt.imshow(drawn.permute(1, 2, 0))
    plt.axis("off")
    plt.show()

    if save_path is not None:
        F.to_pil_image(drawn).save(save_path)
        print(f"結果已存到 {save_path}")

In [ ]:
# 拿測試集前 3 張圖看看
for i in range(3):
    test_img_path = dataset_test.imgs[i]
    predict_and_show(model, test_img_path, device, OUTPUT_DIR / f"prediction_{i}.png")

## 9. 之後直接載入權重推論（不用重新訓練）

權重存在 Drive（`MyDrive/rcnn/outputs/fasterrcnn_vehicles.pth`）的話，重新開 Colab 後：
依序執行第 0～3 節，跳過第 4～7 節的訓練（第 4 節的 `get_model` 定義那格仍要執行），
再執行第 8 節的函式定義那格，最後執行這格即可。

用上傳模式的話，要先把之前下載的 `.pth` 上傳到 `/content/outputs/`。

In [ ]:
model = get_model(NUM_CLASSES)
model.load_state_dict(torch.load(OUTPUT_DIR / "fasterrcnn_vehicles.pth", map_location=device))
model.to(device)

test_img_path = dataset_test.imgs[0]
predict_and_show(model, test_img_path, device)